# Public TCGA WSI processing and UNI-v2 features

Three open TCGA ESCC slides demonstrate image processing only. No trial data, paper-trained parameters, or fictional TCGA treatment outcomes are supplied.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd() if (Path.cwd() / 'helpers').is_dir() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from helpers.io import settings, destination
cfg = settings()
print('Mode:', cfg['mode'])


In [ ]:
import pandas as pd
slides = pd.read_csv(ROOT / 'data/tcga_wsi/manifest.csv')
slides[['patient_id', 'slide_id', 'diagnosis', 'download_url']]

In [ ]:
# Explicit opt-in: downloads approximately 520 MB of public GDC images.
from urllib.request import urlopen
import shutil
DOWNLOAD_WSI = False
if DOWNLOAD_WSI:
    for slide in slides.itertuples():
        target = ROOT / slide.wsi_path
        if target.is_file() and target.stat().st_size == int(slide.file_size):
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        partial = target.with_suffix(target.suffix + '.part')
        with urlopen(slide.download_url, timeout=120) as response, partial.open('wb') as output:
            shutil.copyfileobj(response, output)
        if partial.stat().st_size != int(slide.file_size):
            raise ValueError(f'Incomplete download: {target.name}')
        partial.replace(target)
        print('Downloaded', target.name)
else:
    print('Set DOWNLOAD_WSI=True to download the three open TCGA slides.')

Obtain UNI-v2 access from the official provider. Supply local checkpoint paths through `UNI_V2_WEIGHTS` and, optionally, `HEST_WEIGHTS`. Run extraction on an allocated GPU. The default `wsi_patch_limit=0` uses all segmented tissue coordinates (16,260 across the supplied slides). A positive limit provides an optional sparse preview. Complete verified feature caches are retained in the full local/server project. A fresh Git clone requires downloading the public slides and running feature extraction before notebook 08.

In [ ]:
from helpers.wsi import run_tcga
RUN_WSI = os.environ.get('ESCC_RUN_WSI') == '1'
if RUN_WSI:
    feature_manifest = run_tcga(cfg)
    display(feature_manifest)
else:
    print('Set ESCC_RUN_WSI=1 to enable segmentation, 20x/256px/50% overlap patching and UNI-v2 extraction.')